In [ ]:
try:
    import pyspark.sql.functions as F
    from pyspark.sql import Window
    from pyspark.sql.types import IntegerType
except ModuleNotFoundError as error:
    raise RuntimeError(
        "Este notebook precisa ser executado em um cluster Databricks com PySpark."
    ) from error

SCHEMA_ORIGEM = "bronze"
TABELA_ORIGEM = "tb_movies_info"
SCHEMA_DESTINO = "silver"
TABELA_DESTINO = "tb_info_filmes"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_DESTINO}")

In [ ]:
df_bronze = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_ORIGEM}")

colunas_esperadas = {
    "id", "tconst", "title", "original_title",
    "original_language", "release_date", "runtime",
    "status", "overview", "tagline", "ingestion_datetime"
}
colunas_ausentes = colunas_esperadas.difference(df_bronze.columns)
if colunas_ausentes:
    raise ValueError(f"Colunas ausentes na origem Bronze: {sorted(colunas_ausentes)}")

# mantém a Bronze inalterada e aplica as transformações em uma cópia
df_tratado = (
    df_bronze
    .withColumn("id_filme", F.expr("try_cast(id AS INT)"))
    .withColumn("id_imdb", F.trim(F.col("tconst")))
    .withColumn("titulo", F.trim(F.col("title")))
    .withColumn("titulo_original", F.trim(F.col("original_title")))
    .withColumn("idioma_original", F.trim(F.col("original_language")))
    .withColumn("duracao_minutos", F.expr("try_cast(runtime AS INT)"))
    .withColumn("sinopse", F.trim(F.col("overview")))
    .withColumn("tagline", F.trim(F.col("tagline")))
)

In [ ]:
# normaliza caixa, espaços e hífens antes de traduzir os status
status_normalizado = F.lower(
    F.trim(F.regexp_replace(F.coalesce(F.col("status"), F.lit("")), r"[\s_-]+", " "))
)
df_tratado = df_tratado.withColumn("status_normalizado", status_normalizado)
df_tratado = df_tratado.withColumn(
    "status",
    F.when(F.col("status_normalizado") == "released", F.lit("Lançado"))
     .when(F.col("status_normalizado") == "post production", F.lit("Pós-Produção"))
     .when(F.col("status_normalizado") == "in production", F.lit("Em Produção"))
     .when(F.col("status_normalizado") == "planned", F.lit("Planejado"))
     .otherwise(F.lit("Não Informado"))
).drop("status_normalizado")

In [ ]:
# trata os formatos de data observados; valores inválidos viram NULL
data_texto = F.trim(F.col("release_date"))
data_lancamento = F.coalesce(
    F.expr("try_to_date(release_date, 'yyyy-MM-dd')"),
    F.expr("try_to_date(release_date, 'dd/MM/yyyy')"),
    F.expr("try_to_date(release_date, 'dd-MM-yyyy')"),
    F.expr("try_to_date(release_date, 'MM-dd-yyyy')")
)

df_tratado = (
    df_tratado
    .withColumn("data_lancamento", data_lancamento)
    .withColumn("ano_lancamento", F.year(F.col("data_lancamento")).cast(IntegerType()))
)

# mantém a versão mais recente de cada filme conforme a ingestão
janela_mais_recente = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())
df_tratado = (
    df_tratado
    .withColumn("ordem_ingestao", F.row_number().over(janela_mais_recente))
    .where(F.col("id_filme").isNotNull() & (F.col("ordem_ingestao") == 1))
    .drop("ordem_ingestao", "id", "tconst", "title", "original_title",
          "original_language", "release_date", "runtime", "overview")
)

colunas_silver = [
    "id_filme", "id_imdb", "titulo", "titulo_original",
    "idioma_original", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "status", "sinopse", "tagline",
    "ingestion_datetime"
]
df_silver = df_tratado.select(*colunas_silver)

In [ ]:
# grava a Silver em overwrite para permitir reprocessamento idempotente
(df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_DESTINO}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_DESTINO}")
display(spark.table(f"{SCHEMA_DESTINO}.{TABELA_DESTINO}").limit(10))

In [ ]:
# valida unicidade, tipos tratados e datas não interpretáveis
df_validacao = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DESTINO}")
duplicados = (
    df_validacao.groupBy("id_filme").count().where(F.col("count") > 1).count()
)
datas_invalidas = (
    df_validacao.where(F.col("data_lancamento").isNull()).count()
)

if duplicados != 0:
    raise AssertionError(f"Há {duplicados} ids de filme duplicados na Silver.")

display(
    df_validacao.groupBy("status").count().orderBy(F.col("count").desc())
)
print(f"Registros Silver: {df_validacao.count()}")
print(f"Datas nulas (inválidas ou ausentes): {datas_invalidas}")
print(f"Duplicidades por id_filme: {duplicados}")

In [ ]:
SCHEMA_FINANCEIRO_ORIGEM = "bronze"
TABELA_FINANCEIRO_ORIGEM = "tb_movies_financials"
TABELA_COTACAO_ORIGEM = "tb_cotacao_dolar"
TABELA_FINANCEIRO_DESTINO = "tb_financeiro_filmes"

df_financeiro_bronze = spark.table(f"{SCHEMA_FINANCEIRO_ORIGEM}.{TABELA_FINANCEIRO_ORIGEM}")
colunas_financeiras_esperadas = {"id", "budget", "revenue", "ingestion_datetime"}
colunas_financeiras_ausentes = colunas_financeiras_esperadas.difference(df_financeiro_bronze.columns)
if colunas_financeiras_ausentes:
    raise ValueError(f"Colunas ausentes na origem Bronze: {sorted(colunas_financeiras_ausentes)}")

In [ ]:
# renomeia os campos da Bronze para o padrão da camada Silver sem alterar a origem
df_financeiro = (
    df_financeiro_bronze
    .withColumn("id_filme", F.expr("try_cast(id AS INT)"))
    .withColumn("orcamento_texto", F.trim(F.col("budget")))
    .withColumn("receita_texto", F.trim(F.col("revenue")))
)

# transforma marcadores de ausência em NULL para evitar que textos contaminem os cálculos
marcadores_ausencia = r"^(|unknown|não informado|na|n/a|null)$"
df_financeiro = (
    df_financeiro
    .withColumn("orcamento_texto", F.when(F.lower(F.col("orcamento_texto")).rlike(marcadores_ausencia), F.lit(None)).otherwise(F.col("orcamento_texto")))
    .withColumn("receita_texto", F.when(F.lower(F.col("receita_texto")).rlike(marcadores_ausencia), F.lit(None)).otherwise(F.col("receita_texto")))
)

# remove símbolos monetários e separadores antes do cast; entradas inválidas são convertidas para NULL
df_financeiro = (
    df_financeiro
    .withColumn("orcamento_limpo", F.regexp_replace(F.col("orcamento_texto"), r"[$,\s]", ""))
    .withColumn("receita_limpa", F.regexp_replace(F.col("receita_texto"), r"[$,\s]", ""))
    .withColumn("orcamento_usd", F.expr("try_cast(orcamento_limpo AS DECIMAL(20,2))"))
    .withColumn("receita_usd", F.expr("try_cast(receita_limpa AS DECIMAL(20,2))"))
)

# valores nulos, zerados ou negativos não representam um valor financeiro válido
df_financeiro = (
    df_financeiro
    .withColumn("orcamento_usd", F.when(F.col("orcamento_usd") > 0, F.col("orcamento_usd")))
    .withColumn("receita_usd", F.when(F.col("receita_usd") > 0, F.col("receita_usd")))
)

In [ ]:
df_cotacao = (
    spark.table(f"{SCHEMA_FINANCEIRO_ORIGEM}.{TABELA_COTACAO_ORIGEM}")
    .withColumn("data_hora_cotacao", F.to_timestamp("dataHoraCotacao"))
)

# como a origem financeira não possui data da transação, usa-se a cotação PTAX mais recente disponível
df_cotacao_atual = (
    df_cotacao
    .where(F.col("cotacaoCompra") > 0)
    .orderBy(F.col("data_hora_cotacao").desc())
    .limit(1)
    .select(F.col("cotacaoCompra").cast("DECIMAL(12,6)").alias("cotacao_dolar_brl"))
)

# o cruzamento aplica a mesma taxa de referência a cada filme e mantém o cálculo no cluster
df_financeiro = (
    df_financeiro
    .crossJoin(df_cotacao_atual)
    .withColumn("orcamento_brl", (F.col("orcamento_usd") * F.col("cotacao_dolar_brl")).cast("DECIMAL(20,2)"))
    .withColumn("receita_brl", (F.col("receita_usd") * F.col("cotacao_dolar_brl")).cast("DECIMAL(20,2)"))
    .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast("DECIMAL(20,2)"))
    .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast("DECIMAL(20,2)"))
    .withColumn("margem_lucro_percentual", F.when(F.col("receita_usd") > 0, (F.col("lucro_usd") / F.col("receita_usd") * 100).cast("DECIMAL(10,2)")))
)

In [ ]:
colunas_financeiro_silver = [
    "id_filme", "orcamento_usd", "receita_usd",
    "cotacao_dolar_brl", "orcamento_brl", "receita_brl",
    "lucro_usd", "lucro_brl", "margem_lucro_percentual",
    "ingestion_datetime"
]

# mantém somente a versão mais recente de cada filme após as cargas append da Bronze
janela_financeira_mais_recente = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())
df_financeiro_silver = (
    df_financeiro
    .where(F.col("id_filme").isNotNull())
    .withColumn("ordem_ingestao", F.row_number().over(janela_financeira_mais_recente))
    .where(F.col("ordem_ingestao") == 1)
    .drop("ordem_ingestao")
    .select(*colunas_financeiro_silver)
)

# grava em overwrite para permitir reprocessamento idempotente da tabela Silver
(df_financeiro_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_FINANCEIRO_DESTINO}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_FINANCEIRO_DESTINO}")
display(spark.table(f"{SCHEMA_DESTINO}.{TABELA_FINANCEIRO_DESTINO}").limit(10))

In [ ]:
# valida identificadores, conversões financeiras e ausência de divisão por receita nula
df_financeiro_validacao = spark.table(f"{SCHEMA_DESTINO}.{TABELA_FINANCEIRO_DESTINO}")
duplicados_financeiros = (
    df_financeiro_validacao.groupBy("id_filme").count().where(F.col("count") > 1).count()
)
if duplicados_financeiros != 0:
    raise AssertionError(f"Há {duplicados_financeiros} ids de filme duplicados na Silver financeira.")

display(df_financeiro_validacao.select(
    "id_filme", "orcamento_usd", "receita_usd", "lucro_usd", "margem_lucro_percentual"
).limit(10))
print(f"Registros Silver financeira: {df_financeiro_validacao.count()}")
print(f"Orçamentos válidos: {df_financeiro_validacao.where(F.col("orcamento_usd").isNotNull()).count()}")
print(f"Receitas válidas: {df_financeiro_validacao.where(F.col("receita_usd").isNotNull()).count()}")
print(f"Registros sem receita válida: {df_financeiro_validacao.where(F.col("receita_usd").isNull()).count()}")
print(f"Registros sem orçamento válido: {df_financeiro_validacao.where(F.col("orcamento_usd").isNull()).count()}")
print(f"Duplicidades por id_filme: {duplicados_financeiros}")